# UABR-SLM · Notebook 04 — G1 pilot: does the base model know PopQA facts?

**Phase 04 of 11** (PLAN §11 gate **G1**, amendment A1/A9). GPU **T4 ×1** is enough · ~20–40 min.

**Inputs (Add Input → Your Work):** outputs of notebooks **00**, **01**, **02**, **03** (`model_final.pt`).
**Kaggle settings:** Accelerator **GPU T4** (×1 or ×2) · Internet **On** (to clone the code) · **Save & Run All**.

**Prompt format — ASSUMPTION, team did not decide (change `PROMPT_MODE` / templates before running if you disagree):**
the base model has only seen plain Wikipedia, so it is probed LAMA-style with a **relation cloze template preceded by
3 same-relation TRAIN facts** as demonstrations, e.g.
```
Jaws was directed by Steven Spielberg.
Alien was directed by Ridley Scott.
Heat was directed by Michael Mann.
Hamlet was directed by ▌
```
The **evidence condition** puts the retrieved BM25 windows (notebook 02) before the same prompt. Windows are taken from
the cached top-20 in rank order, **skipping any window that overlaps an already chosen window of the same article**
(team rule), up to 3; then **top-3 if they fit in 512 tokens, else top-2, else top-1** (team rule). Both counts are logged.

**What is measured on the 500 pilot questions (dev-only, never test):**
greedy answer + **k = 8 samples at T = 0.7** (fixed seeds) in both conditions → alias-aware EM / F1 / contains;
**b_quality = mean F1(evidence samples) − mean F1(closed-book samples)** (A1, graded in 1/8 steps);
the likelihood proxy p̄_C − p̄_A as a secondary analysis. Closed-book greedy is also run on **all in-main validation
questions** for more power in the popularity / exposure trends.

**G1 criteria (PLAN §11, A9), as implemented:**
| check | pass condition |
|---|---|
| G1a knowledge beyond the relation prior | pilot closed-book EM − relation-prior EM **≥ 5 pp** and 95 % CI lower bound **> 0** (subject-clustered bootstrap) |
| G1b accuracy rises with popularity or exposure | (top − bottom popularity bin) **or** (High − Control exposure) EM difference has CI lower bound **> 0** (all in-main val) |
| G1c b not almost all positive | share of pilot questions with b_quality > 0 **< 90 %** |
| G1d retrieval damage exists | at least one pilot question correct closed-book but wrong with evidence |

If G1 fails, the plan's fallback applies: re-inject authentic text / more exposure → if still failing, discuss a larger model with the instructor.

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.data import tokenizer as T
from uabr.eval import prompts as PR
from uabr.eval.generate import load_model, encode, fit_evidence, generate, answer_mean_logprob, seed_for
from uabr.eval.metrics import alias_em, alias_f1, alias_contains
from uabr.eval.stats import clustered_bootstrap_mean, clustered_bootstrap_diff
from uabr.retrieval.select import select_non_overlapping

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/04_pilot" if ON_KAGGLE else os.path.abspath("outputs/04_pilot")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

PROMPT_MODE = "cloze_3demo"     # assumption (see header)
N_DEMOS, K, TEMP, MAX_NEW, CTX = 3, 8, 0.7, 12, 512
SEED = 2026
MIN_GAIN_PP = 5.0                # G1a threshold over the relation prior
RUN_FULL_VAL = os.environ.get("RUN_FULL_VAL", "1") == "1"
PILOT_LIMIT = int(os.environ.get("PILOT_LIMIT", 0))   # 0 = all 500 (testing only otherwise)

popqa = pd.read_json(find_input("00_popqa_splits/popqa_prepared.jsonl", "POPQA_PREPARED"), lines=True)
ledger = pd.read_csv(find_input("01_corpus/exposure/ledger_v0.csv", "LEDGER"))
tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20"))
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
MODEL_PATH = find_input("03_pretrain/model_final.pt", "MODEL")
model = load_model(MODEL_PATH, DEV)

WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
TOPDOCS = dict(zip(top["id"], top["doc_ids"]))
WINDOW_WORDS = 60
def evidence_docs(qid):
    return select_non_overlapping(TOPDOCS[qid], WART, WSTART, k=3, window_words=WINDOW_WORDS)
popqa = popqa.merge(ledger[["id", "exposure_group", "subj_found", "obj_in_article", "obj_in_capped"]], on="id", how="left")
DEMOS = PR.pick_demos(popqa[popqa["split"] == "train"], N_DEMOS, seed=SEED)
stop_ids = [i for t, i in tok.get_vocab().items() if any(ch in tok.decode([i]) for ch in "\n.")]
pilot = popqa[popqa["is_pilot"]].reset_index(drop=True)
if PILOT_LIMIT: pilot = pilot.head(PILOT_LIMIT)
assert pilot["in_main"].all() and (pilot["split"] == "val").all()
print(f"pilot {len(pilot)} | model {MODEL_PATH} | device {DEV} | stop tokens {len(stop_ids)}")

## 3 · Prompts used (demonstrations are TRAIN facts)

In [ ]:
for rel in sorted(DEMOS):
    print(f"{rel:15s} demos: {DEMOS[rel]}")
r0 = pilot.iloc[0]
print("\n--- closed-book example ---\n" + PR.closed_prompt(r0.prop, r0.subj, DEMOS) + " ▌")
ws = [WTEXT[d] for d in evidence_docs(r0.id)[0]]
print("\n--- evidence example ---\n" + PR.evidence_prompt(ws, r0.prop, r0.subj, DEMOS) + " ▌")

## 4 · Run the pilot (greedy + 8 samples, both conditions)

In [ ]:
def decode_clean(ids): return PR.clean_generation(tok.decode(ids))

rows = []
for i, r in enumerate(pilot.itertuples(index=False)):
    a_ids = encode(tok, PR.closed_prompt(r.prop, r.subj, DEMOS))
    sel, n_skipped = evidence_docs(r.id)
    ws = [WTEXT[d] for d in sel]
    c_ids, n_used = fit_evidence(tok, lambda w: PR.evidence_prompt(w, r.prop, r.subj, DEMOS), ws, CTX - MAX_NEW)
    rec = {"id": r.id, "evidence_doc_ids": sel[:n_used], "n_overlap_skipped": n_skipped, "n_windows_used": n_used, "prompt_tokens_A": len(a_ids), "prompt_tokens_C": len(c_ids)}
    for cond, ids in [("A", a_ids), ("C", c_ids)]:
        rec[f"greedy_{cond}"] = decode_clean(generate(model, ids, 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0])
        rec[f"samples_{cond}"] = [decode_clean(s) for s in generate(model, ids, K, MAX_NEW, TEMP, seed_for(SEED, r.id, cond), DEV, stop_ids)]
        ans_ids = tok.encode(" " + r.answer, add_special_tokens=False).ids
        rec[f"lp_{cond}"] = answer_mean_logprob(model, ids, ans_ids, DEV)
    rows.append(rec)
    if (i + 1) % 100 == 0: stage(f"  {i+1}/{len(pilot)}")
res = pilot.merge(pd.DataFrame(rows), on="id")
stage(f"done | windows used: {res['n_windows_used'].value_counts().sort_index().to_dict()} | "
      f"questions with >=1 overlapping window skipped: {(res['n_overlap_skipped'] > 0).mean():.0%}")

## 5 · Score: EM / F1 / contains, b_quality (A1), likelihood proxy

In [ ]:
for cond in "AC":
    res[f"em_{cond}"] = [alias_em(p, g) for p, g in zip(res[f"greedy_{cond}"], res["gold_aliases"])]
    res[f"f1_{cond}"] = [alias_f1(p, g) for p, g in zip(res[f"greedy_{cond}"], res["gold_aliases"])]
    res[f"contains_{cond}"] = [alias_contains(p, g) for p, g in zip(res[f"greedy_{cond}"], res["gold_aliases"])]
    res[f"sample_f1_{cond}"] = [float(np.mean([alias_f1(s, g) for s in ss])) for ss, g in zip(res[f"samples_{cond}"], res["gold_aliases"])]
    res[f"pbar_{cond}"] = np.exp(res[f"lp_{cond}"])
res["b_quality"] = res["sample_f1_C"] - res["sample_f1_A"]
res["b_likelihood"] = res["pbar_C"] - res["pbar_A"]
summary = pd.DataFrame({c: {"EM": res[f"em_{c}"].mean(), "F1": res[f"f1_{c}"].mean(), "contains": res[f"contains_{c}"].mean(),
                            "sample_F1": res[f"sample_f1_{c}"].mean()} for c in "AC"}).T
summary.index = ["closed-book (A)", "with evidence (C)"]
summary.loc["relation prior"] = [res["prior_correct"].mean(), np.nan, np.nan, np.nan]
display(summary.round(3))
res[["question", "gold_aliases", "greedy_A", "greedy_C", "em_A", "em_C", "b_quality"]].head(15)

## 6 · G1a — closed-book EM vs relation prior (paired, subject-clustered CI)

In [ ]:
gain = clustered_bootstrap_mean((res["em_A"] - res["prior_correct"]) * 100, res["group_id"])
print(f"closed-book EM {res['em_A'].mean():.1%} | relation prior {res['prior_correct'].mean():.1%} | "
      f"gain {gain['mean']:+.1f} pp [95% CI {gain['lo']:+.1f}, {gain['hi']:+.1f}]")
nonprior = res[res["prior_correct"] == 0]
print(f"EM on questions where the prior answer is wrong (pure knowledge): {nonprior['em_A'].mean():.1%} (n={len(nonprior)})")
by_rel = res.groupby("prop").agg(n=("id", "size"), EM_closed=("em_A", "mean"), prior=("prior_correct", "mean"),
                                 EM_evidence=("em_C", "mean"), contains_closed=("contains_A", "mean"))
by_rel["gain_pp"] = (by_rel["EM_closed"] - by_rel["prior"]) * 100
macro = by_rel["gain_pp"].mean()
print(f"macro-average gain over relations: {macro:+.1f} pp")
display(by_rel.round(3).sort_values("gain_pp"))

## 7 · G1b — does closed-book accuracy rise with popularity and exposure?

In [ ]:
if RUN_FULL_VAL:
    val = popqa[(popqa["split"] == "val") & popqa["in_main"]].reset_index(drop=True)
    preds = []
    for i, r in enumerate(val.itertuples(index=False)):
        ids = encode(tok, PR.closed_prompt(r.prop, r.subj, DEMOS))
        preds.append(decode_clean(generate(model, ids, 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0]))
        if (i + 1) % 500 == 0: stage(f"  val {i+1}/{len(val)}")
    val["greedy_A"] = preds
    val["em_A"] = [alias_em(p, g) for p, g in zip(preds, val["gold_aliases"])]
else:
    val = res.copy()
trend_pop = val.groupby("pop_bin")["em_A"].agg(["mean", "size"]).rename(columns={"mean": "EM"})
trend_exp = val.groupby("exposure_group")["em_A"].agg(["mean", "size"]).rename(columns={"mean": "EM"}).reindex(["control", "low", "medium", "high", "missing"]).dropna()
trend_fact = val.groupby("obj_in_article")["em_A"].agg(["mean", "size"]).rename(columns={"mean": "EM"})
d_pop = clustered_bootstrap_diff(val["em_A"], val["group_id"], val["pop_bin"] == val["pop_bin"].max(), val["pop_bin"] == val["pop_bin"].min())
d_exp = clustered_bootstrap_diff(val["em_A"], val["group_id"], val["exposure_group"] == "high", val["exposure_group"] == "control")
d_prior = clustered_bootstrap_mean((val["em_A"] - val["prior_correct"]) * 100, val["group_id"])
print(f"evaluated on {len(val)} in-main val questions | EM {val['em_A'].mean():.1%} vs prior {val['prior_correct'].mean():.1%} "
      f"(gain {d_prior['mean']:+.1f} pp [{d_prior['lo']:+.1f}, {d_prior['hi']:+.1f}])")
print(f"popularity: top - bottom bin {d_pop['diff']*100:+.1f} pp [{d_pop['lo']*100:+.1f}, {d_pop['hi']*100:+.1f}]")
print(f"exposure:   High - Control   {d_exp['diff']*100:+.1f} pp [{d_exp['lo']*100:+.1f}, {d_exp['hi']*100:+.1f}]")
display(trend_pop.round(3)); display(trend_exp.round(3))
print("EM by whether a gold alias appears in the subject's own article (ledger v0):"); display(trend_fact.round(3))

## 8 · G1c / G1d — benefit distribution, 4-way outcome table, retrieval damage

In [ ]:
b = res["b_quality"]
b_dist = {"P(b>0)": float((b > 0).mean()), "P(b<0)": float((b < 0).mean()), "P(b=0)": float((b == 0).mean()),
          "P(|b|<=0.01)": float((b.abs() <= 0.01).mean()), "P(|b|<=0.05)": float((b.abs() <= 0.05).mean()),
          "P(|b|<=0.10)": float((b.abs() <= 0.10).mean()), "mean_b": float(b.mean())}
print({k: round(v, 3) for k, v in b_dist.items()})
four = pd.crosstab(res["em_A"].map({1.0: "closed correct", 0.0: "closed wrong"}),
                   res["em_C"].map({1.0: "evidence correct", 0.0: "evidence wrong"}))
display(four)
damage = int(((res["em_A"] == 1) & (res["em_C"] == 0)).sum())
print(f"retrieval damage (closed correct -> evidence wrong): {damage} / {int(res['em_A'].sum())} closed-correct")
neither = res[(res["em_A"] == 0) & (res["em_C"] == 0)]
print(f"'neither' questions: {len(neither)} ({len(neither)/len(res):.0%}); of which answer string in top windows: "
      f"{neither.apply(lambda r: any(alias_contains(WTEXT[d], r.gold_aliases) for d in r.evidence_doc_ids), axis=1).mean():.0%}")
from scipy.stats import spearmanr
rho = spearmanr(res["b_quality"], res["b_likelihood"]).correlation
print(f"Spearman(b_quality, likelihood proxy) = {rho:.3f}  (A1 secondary analysis)")
display(res.groupby("pop_bin")["b_quality"].describe()[["mean", "50%", "count"]].round(3))

## 9 · Save

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(b, bins=np.linspace(-1, 1, 33)); ax[0].set(title="b_quality (pilot)", xlabel="mean F1(C) - mean F1(A)")
ax[1].bar(trend_pop.index.astype(str), trend_pop["EM"]); ax[1].set(title="closed-book EM by popularity bin", xlabel="pop bin")
ax[2].bar(trend_exp.index.astype(str), trend_exp["EM"]); ax[2].set(title="closed-book EM by exposure group")
plt.tight_layout(); plt.savefig(f"{OUT}/pilot_overview.png", dpi=120); plt.show()

res.to_parquet(f"{OUT}/pilot_results.parquet", index=False)
val[["id", "prop", "pop_bin", "exposure_group", "greedy_A", "em_A", "prior_correct"]].to_parquet(f"{OUT}/val_closed_book.parquet", index=False)
by_rel.to_csv(f"{OUT}/g1_by_relation.csv"); trend_pop.to_csv(f"{OUT}/g1_by_popbin.csv"); trend_exp.to_csv(f"{OUT}/g1_by_exposure.csv")
four.to_csv(f"{OUT}/four_way_table.csv")
json.dump({"notebook": "04_pilot", "prompt_mode": PROMPT_MODE, "prompt_mode_note": "assumption - team did not decide",
           "templates": PR.CLOZE_TEMPLATES, "demos": DEMOS, "k": K, "temperature": TEMP, "max_new": MAX_NEW, "seed": SEED,
           "evidence_selection": "top-20 rank order, skip same-article overlapping windows, k=3, then overflow 3->2->1",
           "model": MODEL_PATH, "summary": summary.round(4).to_dict(), "gain_pilot_pp": gain, "macro_gain_pp": macro,
           "val_gain_pp": d_prior, "pop_trend": d_pop, "exposure_trend": d_exp, "b_distribution": b_dist,
           "retrieval_damage": damage, "spearman_b_quality_vs_likelihood": rho,
           "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}},
          open(f"{OUT}/meta.json", "w"), indent=2, default=str)

## 10 · Gate G1

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("G1a_gain_over_prior_ge_5pp", gain["mean"] >= MIN_GAIN_PP and gain["lo"] > 0,
    f"{gain['mean']:+.1f} pp [CI {gain['lo']:+.1f}, {gain['hi']:+.1f}] (pilot n={gain['n']})")
add("G1b_rises_with_popularity_or_exposure", d_pop["lo"] > 0 or d_exp["lo"] > 0,
    f"pop top-bottom {d_pop['diff']*100:+.1f} pp (lo {d_pop['lo']*100:+.1f}); High-Control {d_exp['diff']*100:+.1f} pp (lo {d_exp['lo']*100:+.1f})")
add("G1c_b_not_almost_all_positive", b_dist["P(b>0)"] < 0.90, f"P(b>0) = {b_dist['P(b>0)']:.1%}")
add("G1d_retrieval_damage_exists", damage >= 1, f"{damage} questions")
add("info_windows_used", True, f"{res['n_windows_used'].value_counts().sort_index().to_dict()}; overlap-skips in {(res['n_overlap_skipped'] > 0).mean():.0%} of questions", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "G1", "status": status, "checks": checks, "prompt_mode": PROMPT_MODE},
          open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print("\nGATE G1:", status)
if fails:
    print("G1 FAILED -> plan fallback: more authentic exposure / re-injection, then discuss a larger model with the instructor.")

## Next
- **G1 PASS:** notebook 05 (reading preparation, RAFT-style on TRAIN questions) → θ0.
- **G1 FAIL:** send the tables above to the team before anything else; the fallback decision (more exposure vs larger model) changes the plan.